# Concept bottleneck visualizer

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from fastai.vision.all import *
from fastai.learner import Learner
import torch
nn = torch.nn
from settings import *
from utils.learners import *


concepts = pd.read_parquet(root/ "data/semantic_concepts.parquet")
concept_labels = concepts.iloc[:, 4:].columns.tolist()
conceptualized = pd.read_parquet(root/ "data/wikiart_composed_with_concepts.parquet")

# Same imports / helpers as in training notebook
# - concept_labels
# - categories
# - CBMModule class definition
# - get_y_cbm, create_dataloaders_cbm, etc.
# concept_labels, categories, CBMModule, get_y_cbm

In [2]:
from utils.downloader import image_downloader
skip_download = True

if not skip_download:
    # categories = {"depth" : { "symbolic" : ["Mannerism (Late Renaissance)"]} }
    categories, deleted = image_downloader(base="wikiart_composed_with_concepts", categories=categories, sample_size=200, image_size=image_size, skip_downloads=skip_downloads)

    deleted

In [3]:
dataloaders = create_dataloaders_cbm(
    root=container,
    categories=categories,
    dataframe=conceptualized,            # same processed dataframe
    concept_labels=concept_labels,
    bs=32,
    show_batch=False,
    valid_pct=0.2,
    seed=666,
)

In [4]:
def validation_items(dl):
    """Return the image items associated with a fastai DataLoader."""
    
    if hasattr(dl, "items"):
        return list(dl.items)
    
    if hasattr(dl, "dataset") and hasattr(dl.dataset, "items"):
        return list(dl.dataset.items)
    
    raise AttributeError(
        "Could not locate image items. "
        "Expected either dl.items or dl.dataset.items."
    )

In [5]:
for axis in categories:
    train_items = validation_items(dataloaders[axis].train)
    valid_items = validation_items(dataloaders[axis].valid)

    print(f"\n{axis.upper()}")
    print("Training items:", len(train_items))
    print("Validation items:", len(valid_items))
    print("Total:", len(train_items) + len(valid_items))


breadth
Training items: 1798
Validation items: 449
Total: 2247

DEPTH
Training items: 1806
Validation items: 451
Total: 2257


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_choice = "googlenet_10x"

n_concepts = len(concept_labels)
n_classes = 2  # abstract/concrete or iconic/symbolic

learners = {}

for key in categories.keys():
  # Load original GoogLeNet learner (trained on breadth/depth before CBM)
    base_learn = load_learner(garage / model_choice / f"{key}.pkl")
    full_model = base_learn.model           # Sequential with head
    conv_backbone = full_model[0]           # only conv features

    # Recreate CBM
    cbm = CBMModule(conv_backbone, n_concepts=n_concepts, n_classes=n_classes)
    cbm = cbm.to(device)

    # Load CBM weights
    weight_path = garage / model_choice / f"{key}_cbm_weights.pth"
    state = torch.load(weight_path, map_location=device, weights_only=True)
    cbm.load_state_dict(state)
    cbm.eval()

    # Wrap in Learner (optional but convenient)
    loss_func = CBMLoss(concept_weight=1.0, class_weight=1.0)
    learn = Learner(
        dataloaders[key],
        cbm,
        loss_func=loss_func,
        metrics=[cbm_accuracy],
    )
    learn.model.eval()

    learners[key] = learn 

/home/trpquo/miniforge3/envs/fastai/lib/python3.12/site-packages/fastai/learner.py:455: UserWarning: load_learner` uses Python's insecure pickle module, which can execute malicious arbitrary code when loading. Only load files you trust.
If you only need to load model weights and optimizer state, use the safe `Learner.load` instead.
  warn("load_learner` uses Python's insecure pickle module, which can execute malicious arbitrary code when loading. Only load files you trust.\nIf you only need to load model weights and optimizer state, use the safe `Learner.load` instead.")


## Build registries

In [7]:
# Read metadata from conceptuualized
from pathlib import Path
import numpy as np
import pandas as pd
import torch

CLASS_NAMES = {
    "breadth": {0: "abstract", 1: "concrete"},
    "depth":  {0: "iconic",   1: "symbolic"},
}

def get_cbm_learner(axis):
    return learners[axis]

def canonical_content_id(value):
    """Match content IDs regardless of int/string/float representation."""
    if pd.isna(value):
        return None
    
    value = str(value).strip()
    
    if value.endswith(".0"):
        try:
            return str(int(float(value)))
        except ValueError:
            pass
    
    return value

metadata = conceptualized.copy()

if metadata.index.name != "contentId":
    raise ValueError(
        "Expected conceptualized to have a named index 'contentId', "
        f"but its index is named {metadata.index.name!r}."
    )

metadata = metadata.reset_index()

metadata["_content_id_key"] = metadata["contentId"].map(
    canonical_content_id
)

print("contentId is now a normal column:", "contentId" in metadata.columns)
print("Metadata rows:", len(metadata))
print("Unique contentIds:", metadata["_content_id_key"].nunique())

if metadata["_content_id_key"].duplicated().any():
    duplicates = metadata.loc[
        metadata["_content_id_key"].duplicated(keep=False),
        ["contentId", "style", "breadth", "depth"]
    ].sort_values("_content_id_key")
    
    raise ValueError(
        "contentId is not unique in conceptualized. "
        "Resolve these duplicated metadata rows before creating the registry:\n"
        f"{duplicates.head(20)}"
    )

metadata_lookup = metadata.set_index("_content_id_key", drop=False)

print("Metadata rows:", len(metadata_lookup))
print("Concept columns found:", all(col in metadata_lookup.columns for col in concept_labels))

contentId is now a normal column: True
Metadata rows: 39306
Unique contentIds: 39306
Metadata rows: 39306
Concept columns found: True


In [8]:
def build_axis_evaluation_samples(
    axis,
    dataloaders,
    get_learner,
    metadata_lookup,
    concept_labels,
    device,
):
    dl = dataloaders[axis].valid
    model = get_learner(axis).model.to(device).eval()
    items = validation_items(dl)

    records = []
    cursor = 0

    with torch.no_grad():
        for xb, yb in dl:
            y_class, y_concepts = yb
            xb = xb.to(device)

            logits, predicted_concepts = model(xb)

            class_probabilities = torch.softmax(logits, dim=1)
            predicted_class_idx = logits.argmax(dim=1)

            batch_size = xb.shape[0]
            batch_items = items[cursor: cursor + batch_size]

            if len(batch_items) != batch_size:
                raise RuntimeError(
                    f"{axis}: item/batch mismatch at cursor={cursor}. "
                    f"Expected {batch_size} paths but found {len(batch_items)}."
                )

            for row_idx, item in enumerate(batch_items):
                image_path = Path(item)
                content_id = canonical_content_id(image_path.stem)

                if content_id not in metadata_lookup.index:
                    raise KeyError(
                        f"{axis}: contentId {content_id!r} from {image_path} "
                        "was not found in conceptualized."
                    )

                meta = metadata_lookup.loc[content_id]

                record = {
                    "axis": axis,
                    "split": "valid",
                    "contentId": meta["contentId"],
                    "content_id_key": content_id,
                    "image_path": str(image_path),
                    "image_filename": image_path.name,
                    "style": meta["style"],
                    "true_class_idx": int(y_class[row_idx].cpu().item()),
                    "true_class": CLASS_NAMES[axis][
                        int(y_class[row_idx].cpu().item())
                    ],
                    "predicted_class_idx": int(
                        predicted_class_idx[row_idx].cpu().item()
                    ),
                    "predicted_class": CLASS_NAMES[axis][
                        int(predicted_class_idx[row_idx].cpu().item())
                    ],
                    "predicted_class_probability": float(
                        class_probabilities[
                            row_idx,
                            predicted_class_idx[row_idx]
                        ].cpu().item()
                    ),
                    "class_probability_0": float(
                        class_probabilities[row_idx, 0].cpu().item()
                    ),
                    "class_probability_1": float(
                        class_probabilities[row_idx, 1].cpu().item()
                    ),
                }

                for concept_idx, concept_name in enumerate(concept_labels):
                    record[f"true_{concept_name}"] = float(
                        y_concepts[row_idx, concept_idx].cpu().item()
                    )
                    record[f"pred_{concept_name}"] = float(
                        predicted_concepts[row_idx, concept_idx].cpu().item()
                    )

                records.append(record)

            cursor += batch_size

    if cursor != len(items):
        raise RuntimeError(
            f"{axis}: processed {cursor} samples but valid set contains {len(items)} items."
        )

    return pd.DataFrame(records)

In [9]:
evaluation_samples = pd.concat(
    [
        build_axis_evaluation_samples(
            axis=axis,
            dataloaders=dataloaders,
            get_learner=get_cbm_learner,
            metadata_lookup=metadata_lookup,
            concept_labels=concept_labels,
            device=device,
        )
        for axis in list(categories.keys())
    ],
    ignore_index=True,
)

print(evaluation_samples.groupby(["axis", "true_class"]).size())
display(evaluation_samples.head(3))

axis    true_class
breadth  abstract      211
        concrete      238
depth   iconic        266
        symbolic      185
dtype: int64


,axis,split,contentId,content_id_key,image_path,image_filename,style,true_class_idx,true_class,predicted_class_idx,...,true_perspective,pred_perspective,true_dynamic,pred_dynamic,true_balanced,pred_balanced,true_proportional,pred_proportional,true_contained,pred_contained
0,breadth,valid,234585,234585,artefacts/breadth/concrete/234585.jpg,234585.jpg,Baroque,1,concrete,1,...,1.0,0.986178,1.0,0.917272,1.0,0.999988,1.0,0.999877,0.0,0.515226
1,breadth,valid,290268,290268,artefacts/breadth/abstract/290268.jpg,290268.jpg,Abstract Expressionism,0,abstract,0,...,0.0,0.005494,1.0,0.932890,0.0,0.823367,0.0,0.194606,0.0,0.721650
2,breadth,valid,292154,292154,artefacts/breadth/abstract/292154.jpg,292154.jpg,Abstract Expressionism,0,abstract,0,...,0.0,0.004742,1.0,0.886415,0.0,0.088331,0.0,0.047336,0.0,0.297033


In [10]:
# Build the longform evaluation registry
import numpy as np
import pandas as pd

FIXED_COLUMNS = [
    "axis",
    "split",
    "contentId",
    "content_id_key",
    "image_path",
    "image_filename",
    "style",
    "true_class_idx",
    "true_class",
    "predicted_class_idx",
    "predicted_class",
    "predicted_class_probability",
    "class_probability_0",
    "class_probability_1",
]

ARTIFACT_DEFAULTS = {
    "selection_group": None,
    "selection_rank": np.nan,
    "cam_threshold": np.nan,
    "presentation_threshold": np.nan,
    "support_fraction": np.nan,
    "n_support_components": np.nan,
    "largest_component_fraction": np.nan,
    "native_cam_height": np.nan,
    "native_cam_width": np.nan,
    "recomputed_concept_score": np.nan,
    "reconstructed_concept_score": np.nan,
    "concept_logit": np.nan,
    "overlay_path": None,
    "support_mask_path": None,
    "support_overlay_path": None,
    "patch_paths_json": None,
    "n_exported_patches": np.nan,
    "panel_path": None,
    "notes": None,
}

def build_evaluation_registry(evaluation_samples, concept_labels):
    required = [
        *FIXED_COLUMNS,
        *[
            column
            for concept_name in concept_labels
            for column in (
                f"true_{concept_name}",
                f"pred_{concept_name}",
            )
        ],
    ]

    missing = [
        column
        for column in required
        if column not in evaluation_samples.columns
    ]

    if missing:
        raise KeyError(
            "evaluation_samples is missing required columns:\n"
            f"{missing}"
        )

    records = []

    for sample in evaluation_samples.to_dict(orient="records"):
        base = {
            column: sample[column]
            for column in FIXED_COLUMNS
        }

        for concept_idx, concept_name in enumerate(concept_labels):
            records.append({
                **base,
                "concept_idx": int(concept_idx),
                "concept_name": concept_name,
                "true_concept": float(sample[f"true_{concept_name}"]),
                "predicted_concept": float(sample[f"pred_{concept_name}"]),
                **ARTIFACT_DEFAULTS,
            })

    registry = pd.DataFrame(records)

    registry["concept_rank_desc"] = (
        registry
        .groupby(["axis", "concept_idx"])["predicted_concept"]
        .rank(method="first", ascending=False)
        .astype(int)
    )

    expected_rows = len(evaluation_samples) * len(concept_labels)

    assert len(registry) == expected_rows
    assert registry["concept_rank_desc"].notna().all()
    assert registry["image_path"].notna().all()

    return registry

evaluation_registry = build_evaluation_registry(
    evaluation_samples=evaluation_samples,
    concept_labels=concept_labels,
)

print(f"Evaluation images: {len(evaluation_samples)}")
print(f"Concept nodes: {len(concept_labels)}")
print(f"Long-form registry records: {len(evaluation_registry)}")

display(
    evaluation_registry[
        [
            "axis",
            "contentId",
            "style",
            "concept_idx",
            "concept_name",
            "true_concept",
            "predicted_concept",
            "concept_rank_desc",
        ]
    ].head(12)
)

Evaluation images: 900
Concept nodes: 17
Long-form registry records: 15300


,axis,contentId,style,concept_idx,concept_name,true_concept,predicted_concept,concept_rank_desc
0,breadth,234585,Baroque,0,contemporary,0.0,0.001130,419
1,breadth,234585,Baroque,1,figurative,1.0,0.999997,9
2,breadth,234585,Baroque,2,geometrical,0.0,0.000005,442
3,breadth,234585,Baroque,3,pattern,0.0,0.000006,442
4,breadth,234585,Baroque,4,brushstrokes,0.0,0.001968,426
5,breadth,234585,Baroque,5,fixed-pallete,0.0,0.635820,49
6,breadth,234585,Baroque,6,contour,1.0,0.999972,17
7,breadth,234585,Baroque,7,outline,0.0,0.000004,435
8,breadth,234585,Baroque,8,focused,0.5,0.904448,132
9,breadth,234585,Baroque,9,minimalism,0.0,0.233808,273


In [11]:
# save evaluation registries
registry_root = Path("visualizations/googlenet/cbm")
registry_root.mkdir(parents=True, exist_ok=True)

evaluation_samples.to_parquet(
    registry_root / "evaluation_samples_valid.parquet",
    index=False,
)

evaluation_registry.to_parquet(
    registry_root / "evaluation_registry_valid.parquet",
    index=False,
)

print("Saved:")
print(registry_root / "evaluation_samples_valid.parquet")
print(registry_root / "evaluation_registry_valid.parquet")

Saved:
visualizations/googlenet/cbm/evaluation_samples_valid.parquet
visualizations/googlenet/cbm/evaluation_registry_valid.parquet


>>>>>> ?????

In [14]:
# Select endpoint examples

from pathlib import Path

N_PER_CONDITION = 6
AXES = ["breadth", "depth"]

def select_endpoint_examples(
    registry,
    axis,
    concept_idx,
    target_value,
    selection_group,
    n_per_condition=N_PER_CONDITION,
):
    candidates = registry[
        (registry["axis"] == axis)
        & (registry["concept_idx"] == int(concept_idx))
        & np.isclose(
            registry["true_concept"].astype(float),
            float(target_value),
        )
    ].copy()

    candidates["predicted_concept"] = pd.to_numeric(
        candidates["predicted_concept"],
        errors="coerce",
    )

    candidates = (
        candidates
        .dropna(subset=["predicted_concept", "image_path"])
        .sort_values(
            "predicted_concept",
            ascending=False,
            kind="stable",
        )
    )

    if len(candidates) < n_per_condition:
        concept_name = registry.loc[
            registry["concept_idx"] == int(concept_idx),
            "concept_name",
        ].iloc[0]

        return None, {
            "axis": axis,
            "concept_idx": int(concept_idx),
            "concept_name": concept_name,
            "target_value": float(target_value),
            "selection_group": selection_group,
            "n_available": int(len(candidates)),
            "n_required": int(n_per_condition),
            "reason": "insufficient_unambiguous_endpoint_records",
        }

    selected = candidates.head(n_per_condition).copy()

    selected["selection_group"] = selection_group
    selected["selection_rank"] = range(1, n_per_condition + 1)

    return selected, None


def build_endpoint_selection(
    evaluation_registry,
    concept_labels,
    axes=AXES,
    n_per_condition=N_PER_CONDITION,
):
    selected_frames = []
    exclusions = []

    for axis in axes:
        for concept_idx, concept_name in enumerate(concept_labels):

            aligned, aligned_exclusion = select_endpoint_examples(
                registry=evaluation_registry,
                axis=axis,
                concept_idx=concept_idx,
                target_value=1.0,
                selection_group="concept_aligned",
                n_per_condition=n_per_condition,
            )

            contrastive, contrastive_exclusion = select_endpoint_examples(
                registry=evaluation_registry,
                axis=axis,
                concept_idx=concept_idx,
                target_value=0.0,
                selection_group="contrastive_false_positive",
                n_per_condition=n_per_condition,
            )

            if aligned_exclusion is not None:
                exclusions.append(aligned_exclusion)

            if contrastive_exclusion is not None:
                exclusions.append(contrastive_exclusion)

            if aligned is not None and contrastive is not None:
                selected_frames.extend([aligned, contrastive])

    if not selected_frames:
        raise RuntimeError(
            "No complete endpoint panels could be constructed."
        )

    endpoint_selection = (
        pd.concat(selected_frames, ignore_index=True)
        .sort_values(
            [
                "axis",
                "concept_idx",
                "true_concept",
                "selection_rank",
            ],
            ascending=[True, True, False, True],
        )
        .reset_index(drop=True)
    )

    endpoint_exclusions = pd.DataFrame(exclusions)

    complete_panels = (
        endpoint_selection[
            ["axis", "concept_idx", "concept_name"]
        ]
        .drop_duplicates()
        .shape[0]
    )

    expected_records = (
        complete_panels
        * 2
        * n_per_condition
    )

    assert len(endpoint_selection) == expected_records

    return endpoint_selection, endpoint_exclusions


endpoint_selection_df, endpoint_exclusions_df = build_endpoint_selection(
    evaluation_registry=evaluation_registry,
    concept_labels=concept_labels,
)

complete_panels = (
    endpoint_selection_df[
        ["axis", "concept_idx", "concept_name"]
    ]
    .drop_duplicates()
    .shape[0]
)

print(f"Selected endpoint records: {len(endpoint_selection_df)}")
print(f"Complete target-1 versus target-0 panels: {complete_panels}")
print(f"Records per complete panel: {2 * N_PER_CONDITION}")
print(f"Excluded endpoint conditions: {len(endpoint_exclusions_df)}")

if len(endpoint_exclusions_df):
    display(
        endpoint_exclusions_df.sort_values(
            ["axis", "concept_idx", "target_value"]
        )
    )

display(
    endpoint_selection_df[
        [
            "axis",
            "concept_idx",
            "concept_name",
            "true_concept",
            "selection_group",
            "selection_rank",
            "predicted_concept",
            "contentId",
            "image_filename",
        ]
    ].head(24)
)


Selected endpoint records: 384
Complete target-1 versus target-0 panels: 32
Records per complete panel: 12
Excluded endpoint conditions: 2


,axis,concept_idx,concept_name,target_value,selection_group,n_available,n_required,reason
0,breadth,7,outline,1.0,concept_aligned,0,6,insufficient_unambiguous_endpoint_records
1,depth,2,geometrical,1.0,concept_aligned,0,6,insufficient_unambiguous_endpoint_records


,axis,concept_idx,concept_name,true_concept,selection_group,selection_rank,predicted_concept,contentId,image_filename
0,breadth,0,contemporary,1.0,concept_aligned,1,0.999980,287786,287786.jpg
1,breadth,0,contemporary,1.0,concept_aligned,2,0.999958,323563,323563.jpg
2,breadth,0,contemporary,1.0,concept_aligned,3,0.999943,329610,329610.jpg
3,breadth,0,contemporary,1.0,concept_aligned,4,0.999930,325780,325780.jpg
4,breadth,0,contemporary,1.0,concept_aligned,5,0.999927,286340,286340.jpg
5,breadth,0,contemporary,1.0,concept_aligned,6,0.999913,309991,309991.jpg
6,breadth,0,contemporary,0.0,contrastive_false_positive,1,0.986617,248930,248930.jpg
7,breadth,0,contemporary,0.0,contrastive_false_positive,2,0.974496,195402,195402.jpg
8,breadth,0,contemporary,0.0,contrastive_false_positive,3,0.970361,195418,195418.jpg
9,breadth,0,contemporary,0.0,contrastive_false_positive,4,0.939454,9223372032559856650,9223372032559856650.jpg


In [15]:

output_dir = Path(
    "visualizations/googlenet/cbm_concept_cam"
)
output_dir.mkdir(parents=True, exist_ok=True)

evaluation_registry.to_parquet(
    output_dir / "evaluation_registry.parquet",
    index=False,
)

endpoint_selection_df.to_parquet(
    output_dir / "endpoint_selection.parquet",
    index=False,
)

endpoint_exclusions_df.to_parquet(
    output_dir / "endpoint_exclusions.parquet",
    index=False,
)

## Concept-CAM visualization (**Concept Activation Map**)

GoogLeNet CBM has the following relevant structure:

> Inception5b feature maps → global average pooling →  linear concept head → $\sigma(\cdot)$

For concept $c$, the pre-sigmoid concept evidence is:

$$z_c=b_c+ \sum_k w_{c,k} \biggl( \frac{1}{HW} \sum_{ij}A_k(i,j)\bigr).$$

Therefore we can project the learned concept-head weights $w_{c,k}$ back onto the final Inception5b feature maps:

$$M_c(i,j)= \sum_k w_{c,k}A_k(i,j).$$

This produces a genuine spatial concept-evidence map. Its spatial mean—together with the concept-head bias—is exactly the pre-sigmoid logit for that concept. It is therefore more directly connected to the concept node than Grad-CAM, which estimates feature-map importance from gradients. CAM was introduced for CNNs with global average pooling followed by a linear classifier; it produces localization maps by applying output weights directly to the final convolutional feature maps.[@Zhou2016]


In [16]:
# Concept_CAM Class definition

import torch
import torch.nn.functional as F

class ConceptCAM:
    """
    Exact CAM for a CBM concept head:
    Inception5b -> global average pooling -> Linear(1024, n_concepts) -> sigmoid.
    """

    def __init__(self, model, target_layer, concept_head):
        self.model = model.eval()
        self.target_layer = target_layer
        self.concept_head = concept_head
        self.activations = None

        self.hook_handle = target_layer.register_forward_hook(
            self._capture_activations
        )

    def _capture_activations(self, module, inputs, output):
        if not torch.is_tensor(output):
            raise TypeError("The CAM target layer must return a tensor.")
        self.activations = output

    def remove_hook(self):
        self.hook_handle.remove()

    @torch.no_grad()
    def __call__(self, xb, concept_idx):
        self.activations = None

        logits, predicted_concepts = self.model(xb)

        if self.activations is None:
            raise RuntimeError(
                "No Inception5b activation was captured by the hook."
            )

        activations = self.activations
        concept_weights = self.concept_head.weight[concept_idx]

        if activations.shape[1] != concept_weights.numel():
            raise ValueError(
                f"Feature-channel mismatch: Inception5b has "
                f"{activations.shape[1]} channels but concept {concept_idx} "
                f"has {concept_weights.numel()} head weights."
            )

        signed_cam = torch.einsum(
            "k,bkhw->bhw",
            concept_weights,
            activations,
        )

        positive_cam = F.relu(signed_cam)

        pooled_features = activations.mean(dim=(2, 3))
        reconstructed_logits = F.linear(
            pooled_features,
            self.concept_head.weight,
            self.concept_head.bias,
        )

        predicted_scores = predicted_concepts[:, concept_idx]
        reconstructed_score = torch.sigmoid(
            reconstructed_logits[:, concept_idx]
        )

        return {
            "signed_cam": signed_cam.detach(),
            "positive_cam": positive_cam.detach(),
            "concept_score": predicted_scores.detach(),
            "reconstructed_concept_score": reconstructed_score.detach(),
            "concept_logit": reconstructed_logits[:, concept_idx].detach(),
            "inception_shape": tuple(activations.shape),
            "class_logits": logits.detach(),
        }

def normalize_map(values, eps=1e-8):
    values = np.asarray(values, dtype=np.float32)
    low = values.min()
    high = values.max()
    return (values - low) / max(high - low, eps)

### Adaptive support-region procedure

In [22]:
# Rendering helpers definitions

from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from PIL import Image
from scipy import ndimage as ndi
from matplotlib import cm

try:
    BILINEAR = Image.Resampling.BILINEAR
    NEAREST = Image.Resampling.NEAREST
except AttributeError:
    BILINEAR = Image.BILINEAR
    NEAREST = Image.NEAREST

#### Batch process concept-CAM evidence panels

In [19]:
# Define a function to create exemplar panel per concept

def save_concept_exemplar_panel(
    selected_rows,
    output_root,
    axis,
    concept_idx,
    concept_name,
    ncols=6,
):
    concept_slug = safe_filename(concept_name)

    panel_path = (
        output_root
        / "panels"
        / axis
        / f"concept-{concept_idx:02d}_{concept_slug}_topk.png"
    )

    panel_path.parent.mkdir(parents=True, exist_ok=True)

    rows = selected_rows.sort_values("selection_rank")
    n_images = len(rows)
    nrows = int(np.ceil(n_images / ncols))

    if n_images < 1:
        return None
    
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(5 * ncols, 5 * nrows),
    )

    axes = np.asarray(axes).reshape(-1)

    for ax, (_, row) in zip(axes, rows.iterrows()):
        overlay = Image.open(row["overlay_path"]).convert("RGB")

        ax.imshow(overlay)
        ax.set_title(
            f"#{int(row['selection_rank'])} | "
            f"pred={row['predicted_concept']:.3f} | "
            f"true={row['true_concept']}\n"
            f"{row['style']}",
            fontsize=9,
        )
        ax.axis("off")

    for ax in axes[n_images:]:
        ax.axis("off")

    fig.suptitle(
        f"{axis.capitalize()} CBM — concept {concept_idx}: {concept_name}",
        fontsize=15,
        y=0.995,
    )

    plt.tight_layout()
    fig.savefig(
        panel_path,
        dpi=180,
        bbox_inches="tight",
    )
    plt.close(fig)

    return str(panel_path)

In [20]:
# Define helper functions
from pathlib import Path
import re

from PIL import Image
from scipy import ndimage as ndi
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch


def safe_filename(value, max_length=80):
    value = str(value).strip().lower()
    value = re.sub(r"[^a-z0-9]+", "-", value)
    value = value.strip("-")
    return value[:max_length] or "unnamed"


def make_support_overlay(
    image_rgb,
    presentation_support,
    fill_alpha=0.45,
    fill_color=(1.0, 0.15, 0.65),
    boundary_color=(1.0, 1.0, 0.0),
):
    """
    Render smooth presentation support as a colored blob plus yellow boundary.

    This is display-only. Analytical component statistics remain based on the
    native 8x8 support mask.
    """
    base = np.asarray(image_rgb, dtype=np.float32) / 255.0
    mask = np.asarray(presentation_support, dtype=bool)

    output = base.copy()

    fill_color = np.asarray(fill_color, dtype=np.float32)
    output[mask] = (
        (1.0 - fill_alpha) * output[mask]
        + fill_alpha * fill_color
    )

    boundary = mask & ~ndi.binary_erosion(
        mask,
        structure=np.ones((3, 3), dtype=bool),
    )

    output[boundary] = np.asarray(boundary_color, dtype=np.float32)

    return np.clip(output, 0.0, 1.0)


def save_rgb_image(image_array, output_path):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    image_array = np.asarray(image_array)

    if image_array.dtype != np.uint8:
        image_array = np.clip(image_array * 255.0, 0, 255).astype(np.uint8)

    Image.fromarray(image_array).save(output_path)
    return str(output_path)


def save_binary_mask(mask, output_path):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    mask_image = (
        np.asarray(mask, dtype=np.uint8) * 255
    )

    Image.fromarray(mask_image, mode="L").save(output_path)
    return str(output_path)

In [21]:
# More helper functions ???

def resize_map(cam, image_size, interpolation=BILINEAR):
    """
    Resize one native-resolution CAM or binary mask to (width, height).
    """
    cam = np.asarray(cam, dtype=np.float32)
    image = Image.fromarray(cam, mode="F")
    return np.asarray(
        image.resize(image_size, resample=interpolation),
        dtype=np.float32,
    )


def normalize_positive_cam(cam, eps=1e-8):
    """
    Normalize positive evidence for display only.
    """
    cam = np.maximum(np.asarray(cam, dtype=np.float32), 0.0)
    max_value = cam.max()

    if max_value <= eps:
        return np.zeros_like(cam)

    return cam / max_value


def extract_adaptive_support(
    positive_cam,
    quantile=0.85,
    min_component_cells=1,
):
    """
    Derive a binary support mask from native-resolution positive CAM evidence.

    Returns the threshold, native binary mask, component labels,
    and component-level native-grid measurements.
    """
    positive_cam = np.maximum(
        np.asarray(positive_cam, dtype=np.float32),
        0.0,
    )

    positive_values = positive_cam[positive_cam > 0]

    if positive_values.size == 0:
        empty = np.zeros_like(positive_cam, dtype=bool)
        return {
            "threshold": np.nan,
            "mask": empty,
            "labels": np.zeros_like(positive_cam, dtype=np.int32),
            "components": [],
        }

    threshold = float(np.quantile(positive_values, quantile))
    mask = positive_cam >= threshold

    labels, n_labels = ndi.label(
        mask,
        structure=np.ones((3, 3), dtype=np.int8),
    )

    components = []
    retained_mask = np.zeros_like(mask, dtype=bool)
    retained_labels = np.zeros_like(labels, dtype=np.int32)

    new_label = 0

    for old_label in range(1, n_labels + 1):
        component_mask = labels == old_label
        area_cells = int(component_mask.sum())

        if area_cells < min_component_cells:
            continue

        new_label += 1
        retained_mask |= component_mask
        retained_labels[component_mask] = new_label

        rows, cols = np.where(component_mask)

        components.append({
            "component_id": new_label,
            "area_cells": area_cells,
            "row_min": int(rows.min()),
            "row_max": int(rows.max()),
            "col_min": int(cols.min()),
            "col_max": int(cols.max()),
            "mean_evidence": float(positive_cam[component_mask].mean()),
            "max_evidence": float(positive_cam[component_mask].max()),
        })

    return {
        "threshold": threshold,
        "mask": retained_mask,
        "labels": retained_labels,
        "components": components,
    }


def component_crop_box(
    component,
    cam_shape,
    image_size,
    padding_fraction=0.10,
):
    """
    Convert one native CAM-grid component into a padded image-space crop box.

    Returns Pillow-compatible bounds: (left, top, right, bottom).
    """
    cam_h, cam_w = cam_shape
    image_w, image_h = image_size

    left = component["col_min"] * image_w / cam_w
    right = (component["col_max"] + 1) * image_w / cam_w
    top = component["row_min"] * image_h / cam_h
    bottom = (component["row_max"] + 1) * image_h / cam_h

    width = right - left
    height = bottom - top

    pad_x = padding_fraction * width
    pad_y = padding_fraction * height

    left = max(0, int(np.floor(left - pad_x)))
    top = max(0, int(np.floor(top - pad_y)))
    right = min(image_w, int(np.ceil(right + pad_x)))
    bottom = min(image_h, int(np.ceil(bottom + pad_y)))

    return left, top, right, bottom


def make_positive_overlay(
    image_rgb,
    positive_cam_resized,
    alpha=0.75,
    gamma=1.6,
    cmap_name="magma",
):
    """
    Blend positive concept evidence with the source painting.

    gamma > 1 reduces low-evidence coloration and emphasizes high-evidence
    regions; alpha controls the maximum opacity at the strongest locations.
    """
    image_rgb = np.asarray(image_rgb, dtype=np.float32) / 255.0
    cam_norm = normalize_positive_cam(positive_cam_resized)

    heatmap_rgb = plt.colormaps[cmap_name](cam_norm)[..., :3]

    alpha_map = alpha * np.power(cam_norm, gamma)[..., None]

    overlay = (
        (1.0 - alpha_map) * image_rgb
        + alpha_map * heatmap_rgb
    )

    return np.clip(overlay, 0.0, 1.0)

## Create "concept-aligned vs contrastive" panels per concept per axis

In [23]:
import pandas as pd

# load manifests
selection_root = Path(
    "visualizations/googlenet/cbm_concept_cam"
)

endpoint_selection_df = pd.read_parquet(
    selection_root / "endpoint_selection.parquet",
)

endpoint_exclusions_df = pd.read_parquet(
    selection_root / "endpoint_exclusions.parquet",
)

evaluation_registry = endpoint_selection_df.copy()

In [24]:
# Panel functions

PANEL_ROOT = Path("visualizations/googlenet/cbm_concept_panels")
PANEL_ROOT.mkdir(parents=True, exist_ok=True)

N_PER_ROW = 6
RED_RGB = np.array([255, 0, 0], dtype=np.uint8)
RED_ALPHA = 0.52


def safe_filename(value):
    value = str(value).strip().lower()
    value = re.sub(r"[^\w.-]+", "_", value)
    return re.sub(r"_+", "_", value).strip("_")


def load_red_support_overlay(image_path, mask_path, alpha=RED_ALPHA):
    image = Image.open(image_path).convert("RGB")
    mask = Image.open(mask_path).convert("L")

    if mask.size != image.size:
        mask = mask.resize(image.size, Image.Resampling.NEAREST)

    image_arr = np.asarray(image).copy()
    mask_arr = np.asarray(mask)

    support = mask_arr > 0
    image_arr[support] = (
        (1 - alpha) * image_arr[support] + alpha * RED_RGB
    ).astype(np.uint8)

    return Image.fromarray(image_arr)


def select_concept_examples(
    registry_df,
    axis,
    concept_idx,
    target_value,
    n=N_PER_ROW,
):
    cols = REGISTRY_COLUMNS

    selected = registry_df[
        (registry_df[cols["axis"]] == axis)
        & (registry_df[cols["concept_idx"]].astype(int) == int(concept_idx))
        & (registry_df[cols["target"]].astype(float) == float(target_value))
    ].copy()

    selected[cols["predicted_score"]] = pd.to_numeric(
        selected[cols["predicted_score"]],
        errors="coerce",
    )

    selected = selected.dropna(
        subset=[
            cols["predicted_score"],
            cols["image_path"],
            cols["support_mask_path"],
        ]
    )

    selected = selected.sort_values(
        cols["predicted_score"],
        ascending=False,
    ).head(n)

    if len(selected) < n:
        raise ValueError(
            f"{axis}, concept index {concept_idx}, target={target_value}: "
            f"found only {len(selected)} usable records; expected {n}."
        )

    return selected


def save_concept_aligned_contrastive_panel(
    registry_df,
    axis,
    concept_idx,
    concept_name,
    output_root=PANEL_ROOT,
    n=N_PER_ROW,
    dpi=180,
):
    cols = REGISTRY_COLUMNS

    aligned = select_concept_examples(
        registry_df=registry_df,
        axis=axis,
        concept_idx=concept_idx,
        target_value=1,
        n=n,
    )

    contrastive = select_concept_examples(
        registry_df=registry_df,
        axis=axis,
        concept_idx=concept_idx,
        target_value=0,
        n=n,
    )

    fig, axes = plt.subplots(
        nrows=2,
        ncols=n,
        figsize=(3.0 * n, 6.5),
        constrained_layout=False,
    )

    row_specifications = [
        (
            aligned,
            "Concept-aligned exemplars (target = 1)",
        ),
        (
            contrastive,
            "Contrastive exemplars (target = 0)",
        ),
    ]

    for row_idx, (rows, row_title) in enumerate(row_specifications):
        for col_idx, (_, record) in enumerate(rows.iterrows()):
            ax = axes[row_idx, col_idx]

            overlay = load_red_support_overlay(
                image_path=record[cols["image_path"]],
                mask_path=record[cols["support_mask_path"]],
            )

            ax.imshow(overlay)
            ax.axis("off")

            score = float(record[cols["predicted_score"]])
            ax.set_title(
                f"target={int(record[cols['target']])} | "
                f"predicted={score:.3f}",
                fontsize=9,
                pad=5,
            )

        axes[row_idx, 0].set_ylabel(
            row_title,
            fontsize=12,
            fontweight="bold",
            rotation=90,
            labelpad=18,
        )

    fig.suptitle(
        f"{axis.title()} CBM — {concept_name}\n"
        "Adaptive concept-conditioned support regions",
        fontsize=15,
        fontweight="bold",
        y=0.995,
    )

    output_dir = Path(output_root) / axis
    output_dir.mkdir(parents=True, exist_ok=True)

    output_path = output_dir / (
        f"{int(concept_idx):02d}_{safe_filename(concept_name)}_"
        "concept-aligned_vs_contrastive.png"
    )

    fig.savefig(
        output_path,
        dpi=dpi,
        bbox_inches="tight",
        facecolor="white",
    )
    plt.close(fig)

    return output_path

In [25]:
# Define the exporter function

import json
from pathlib import Path

import numpy as np
from PIL import Image

RED_RGB = np.array([255, 0, 0], dtype=np.uint8)


def make_red_support_overlay(
    image_rgb,
    support_mask,
    fill_alpha=0.42,
):
    image_rgb = np.asarray(image_rgb).copy()

    if support_mask.shape != image_rgb.shape[:2]:
        raise ValueError(
            f"Mask shape {support_mask.shape} does not match "
            f"image shape {image_rgb.shape[:2]}"
        )

    support_mask = support_mask.astype(bool)

    image_rgb[support_mask] = (
        (1.0 - fill_alpha) * image_rgb[support_mask]
        + fill_alpha * RED_RGB
    ).astype(np.uint8)

    return image_rgb


def process_concept_exemplar(
    row,
    model,
    concept_cam,
    axis,
    concept_name,
    output_root,
    input_size=(256, 256),
    support_quantile=0.85,
    blur_sigma=2.5,
    crop_padding_fraction=0.10,
):
    concept_idx = int(row["concept_idx"])
    rank = int(row["selection_rank"])
    content_id = row["contentId"]
    selection_group = safe_filename(row["selection_group"])

    image_path = Path(row["image_path"])

    test_dl = dataloaders[axis].test_dl(
        [image_path],
        with_labels=False,
        shuffle=False,
    )

    xb = test_dl.one_batch()[0].to(device)

    result = concept_cam(xb, concept_idx)

    positive_cam = result["positive_cam"][0].cpu().numpy()

    input_h, input_w = xb.shape[-2:]
    image_size = (input_w, input_h)

    original = Image.open(image_path).convert("RGB")
    display_image = original.resize(image_size, resample=BILINEAR)
    display_rgb = np.asarray(display_image)

    support = extract_adaptive_support(
        positive_cam,
        quantile=support_quantile,
        min_component_cells=1,
    )

    positive_cam_resized = resize_map(
        positive_cam,
        image_size=image_size,
        interpolation=BILINEAR,
    )

    presentation_evidence = ndi.gaussian_filter(
        positive_cam_resized,
        sigma=blur_sigma,
    )

    nonzero_presentation = presentation_evidence[
        presentation_evidence > 0
    ]

    if nonzero_presentation.size:
        presentation_threshold = float(
            np.quantile(
                nonzero_presentation,
                support_quantile,
            )
        )
    else:
        presentation_threshold = np.nan

    native_mask_resized = resize_map(
        support["mask"].astype(np.float32),
        image_size=image_size,
        interpolation=NEAREST,
    ).astype(bool)

    evidence_overlay = make_positive_overlay(
        display_rgb,
        positive_cam_resized,
        alpha=0.75,
        gamma=1.6,
        cmap_name="magma",
    )

    red_support_overlay = make_red_support_overlay(
        display_rgb,
        native_mask_resized,
        fill_alpha=0.42,
    )

    concept_slug = safe_filename(concept_name)

    base_name = (
        f"concept-{concept_idx:02d}_{concept_slug}"
        f"_{selection_group}"
        f"_rank-{rank:02d}"
        f"_content-{content_id}"
        f"_score-{row['predicted_concept']:.4f}"
    )

    overlay_path = (
        output_root / "overlays" / axis / f"{base_name}.png"
    )

    support_overlay_path = (
        output_root / "support_overlays" / axis / f"{base_name}.png"
    )

    support_mask_path = (
        output_root / "support_masks" / axis / f"{base_name}.png"
    )

    patch_dir = output_root / "patches" / axis

    for directory in [
        overlay_path.parent,
        support_overlay_path.parent,
        support_mask_path.parent,
        patch_dir,
    ]:
        directory.mkdir(parents=True, exist_ok=True)

    save_rgb_image(evidence_overlay, overlay_path)
    save_rgb_image(red_support_overlay, support_overlay_path)
    save_binary_mask(native_mask_resized, support_mask_path)

    patch_paths = []

    for component in support["components"]:
        crop_box = component_crop_box(
            component=component,
            cam_shape=positive_cam.shape,
            image_size=image_size,
            padding_fraction=crop_padding_fraction,
        )

        patch = display_image.crop(crop_box)

        patch_path = (
            patch_dir
            / (
                f"{base_name}"
                f"_component-{component['component_id']:02d}.png"
            )
        )

        patch.save(patch_path)
        patch_paths.append(str(patch_path))

    native_support_cells = int(support["mask"].sum())

    if native_support_cells > 0:
        largest_component_fraction = (
            max(
                component["area_cells"]
                for component in support["components"]
            )
            / native_support_cells
        )
    else:
        largest_component_fraction = 0.0

    return {
        "overlay_path": str(overlay_path),
        "support_overlay_path": str(support_overlay_path),
        "support_mask_path": str(support_mask_path),
        "patch_paths_json": json.dumps(patch_paths),
        "n_exported_patches": len(patch_paths),
        "cam_threshold": float(support["threshold"]),
        "presentation_threshold": presentation_threshold,
        "support_fraction": float(support["mask"].mean()),
        "n_support_components": len(support["components"]),
        "largest_component_fraction": float(
            largest_component_fraction
        ),
        "native_cam_height": int(positive_cam.shape[0]),
        "native_cam_width": int(positive_cam.shape[1]),
        "recomputed_concept_score": float(
            result["concept_score"][0].cpu().item()
        ),
        "reconstructed_concept_score": float(
            result["reconstructed_concept_score"][0].cpu().item()
        ),
        "concept_logit": float(
            result["concept_logit"][0].cpu().item()
        ),
    }

In [22]:
# FULL EXPORT : Run for all 17 concepts for both axis
TOP_K = 9
SUPPORT_QUANTILE = 0.85

concept_cam = {}

for axis in categories:


    output_root = Path(
        "visualizations/googlenet/cbm_concept_cam"
    )

    model = learners[axis].model.to(device).eval()

    concept_cam[axis] = ConceptCAM(
        model=model,
        target_layer=model.backbone[15],
        concept_head=model.concept_head,
    )

    for concept_idx, concept_name in enumerate(concept_labels):
        print(
            f"Processing {axis}: "
            f"concept {concept_idx:02d} — {concept_name}"
        )

        selected_idx = (
            evaluation_registry[
                (evaluation_registry["axis"] == axis)
                & (evaluation_registry["concept_idx"] == concept_idx)
            ]
            .sort_values("predicted_concept", ascending=False)
            # .head(TOP_K)
            .index
        )

        evaluation_registry.loc[
            selected_idx,
            "selection_group"
        ] = "top_predicted"

        evaluation_registry.loc[
            selected_idx,
            "selection_rank"
        ] = np.arange(1, len(selected_idx) + 1)

        for registry_idx in selected_idx:
            row = evaluation_registry.loc[registry_idx]

            artifact_data = process_concept_exemplar(
                row=row,
                model=model,
                concept_cam=concept_cam[axis],
                axis=axis,
                concept_name=concept_name,
                output_root=output_root,
                support_quantile=SUPPORT_QUANTILE,
            )

            for column, value in artifact_data.items():
                if column not in evaluation_registry.columns:
                    evaluation_registry[column] = None
                
                evaluation_registry.at[registry_idx, column] = value

        concept_rows = evaluation_registry.loc[selected_idx].copy()

        panel_path = save_concept_exemplar_panel(
            selected_rows=concept_rows,
            output_root=output_root,
            axis=axis,
            concept_idx=concept_idx,
            concept_name=concept_name,
            ncols=3,
        )

        evaluation_registry.loc[
            selected_idx,
            "panel_path"
        ] = panel_path

    concept_cam[axis].remove_hook()

    print(f"\nCompleted {axis} concept-CAM export.\n\n")

Processing breadth: concept 00 — contemporary
Processing breadth: concept 01 — figurative
Processing breadth: concept 02 — geometrical
Processing breadth: concept 03 — pattern
Processing breadth: concept 04 — brushstrokes
Processing breadth: concept 05 — fixed-pallete
Processing breadth: concept 06 — contour
Processing breadth: concept 07 — outline
Processing breadth: concept 08 — focused
Processing breadth: concept 09 — minimalism
Processing breadth: concept 10 — texture
Processing breadth: concept 11 — isometric
Processing breadth: concept 12 — perspective
Processing breadth: concept 13 — dynamic
Processing breadth: concept 14 — balanced
Processing breadth: concept 15 — proportional
Processing breadth: concept 16 — contained

Completed breadth concept-CAM export.


Processing depth: concept 00 — contemporary
Processing depth: concept 01 — figurative
Processing depth: concept 02 — geometrical
Processing depth: concept 03 — pattern
Processing depth: concept 04 — brushstrokes
Processing

In [26]:
for axis in categories:
    exported = evaluation_registry[
        (evaluation_registry["axis"] == axis)
        & (evaluation_registry["selection_group"] == "top_predicted")
    ].copy()

    print(f"Selected {axis} rows:", len(exported))
    print("Expected selected rows:", len(concept_labels) * TOP_K)

    print(
        "Rows with overlay files:",
        exported["overlay_path"].notna().sum()
    )

    print(
        "Rows with support-mask files:",
        exported["support_mask_path"].notna().sum()
    )

    print(
        "Total exported component patches:",
        exported["n_exported_patches"].sum()
    )

    # assert len(exported) == len(concept_labels) * TOP_K
    assert exported["overlay_path"].notna().all()
    assert exported["support_mask_path"].notna().all()
    assert exported["patch_paths_json"].notna().all()

Selected breadth rows: 0


NameError: name 'TOP_K' is not defined

In [27]:
print("Objects with 'cam' in their global name:")

for name in sorted(globals()):
    if "cam" in name.lower():
        value = globals()[name]
        print(f"{name}: {type(value).__name__}")

Objects with 'cam' in their global name:
ConceptCAM: type
camel2snake: function
camel2words: function
normalize_positive_cam: function
snake2camel: function


In [28]:
for axis in ["breadth", "depth"]:
    model = learners[axis].model

    print(f"\n{axis.upper()}")
    print("Model id:", id(model))

    for attr_name in ["model", "target_model", "net"]:
        if hasattr(concept_cam, attr_name):
            cam_model = getattr(concept_cam, attr_name)

            print(f"concept_cam.{attr_name} id:", id(cam_model))
            print(
                f"Matches {axis} model:",
                cam_model is model,
            )


breadth
Model id: 126803800973360


NameError: name 'concept_cam' is not defined

In [26]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# ------------------------------------------------------------------
# INPUT TABLE
# evaluation_registry may be a DataFrame already, or a dict/list of records.
# ------------------------------------------------------------------

if isinstance(evaluation_registry, pd.DataFrame):
    panel_source = evaluation_registry.copy()
elif isinstance(evaluation_registry, dict):
    panel_source = pd.DataFrame(
        record
        for records in evaluation_registry.values()
        for record in records
    )
else:
    panel_source = pd.DataFrame(evaluation_registry)

# ------------------------------------------------------------------
# ADAPT THESE ONLY IF YOUR REGISTRY USES DIFFERENT COLUMN NAMES
# ------------------------------------------------------------------
AXIS_COL = "axis"
CONCEPT_IDX_COL = "concept_idx"
CONCEPT_NAME_COL = "concept_name"
TRUE_CONCEPT_COL = "true_concept"
PRED_CONCEPT_COL = "predicted_concept"
IMAGE_PATH_COL = "image_path"
SUPPORT_MASK_COL = "support_mask_path"

# If your actual columns differ, inspect:
# print(panel_source.columns.tolist())

required_columns = [
    AXIS_COL,
    CONCEPT_IDX_COL,
    CONCEPT_NAME_COL,
    TRUE_CONCEPT_COL,
    PRED_CONCEPT_COL,
    IMAGE_PATH_COL,
    SUPPORT_MASK_COL,
]
missing_columns = [c for c in required_columns if c not in panel_source.columns]

if missing_columns:
    raise KeyError(
        "The registry does not contain these required columns:\n"
        f"{missing_columns}\n\n"
        "Available columns:\n"
        f"{panel_source.columns.tolist()}\n\n"
        "Update the *_COL variables above to match the registry."
    )

# ------------------------------------------------------------------
# IMAGE / OVERLAY HELPERS
# ------------------------------------------------------------------
def load_rgb_image(path):
    return Image.open(path).convert("RGB")

def load_mask(path, size):
    mask = Image.open(path).convert("L")
    if mask.size != size:
        mask = mask.resize(size, resample=Image.Resampling.NEAREST)
    return np.asarray(mask) > 0

def red_support_overlay(image, support_mask, alpha=0.52):
    """
    Red transparent overlay, no contour.
    `image`: PIL RGB image
    `support_mask`: Boolean array matching image height/width
    """
    base = np.asarray(image).astype(np.float32)
    out = base.copy()

    red = np.zeros_like(base)
    red[..., 0] = 255

    out[support_mask] = (
        (1 - alpha) * base[support_mask]
        + alpha * red[support_mask]
    )

    return np.clip(out, 0, 255).astype(np.uint8)

def safe_name(value):
    return (
        str(value)
        .strip()
        .lower()
        .replace(" ", "_")
        .replace("/", "-")
        .replace("\\", "-")
    )

# ------------------------------------------------------------------
# PANEL CREATION
# ------------------------------------------------------------------
PANEL_ROOT = Path(
    "visualizations/googlenet/"
    "concept_aligned_contrastive_panel"
)

N_PER_ROW = 6
FIGSIZE_PER_TILE = (3.2, 3.4)
OVERLAY_ALPHA = 0.52

panel_export_records = []
panel_summary_records = []

for axis in sorted(panel_source[AXIS_COL].dropna().unique()):
    axis_df = panel_source[panel_source[AXIS_COL] == axis].copy()

    for concept_idx in sorted(axis_df[CONCEPT_IDX_COL].dropna().unique()):
        concept_df = axis_df[
            axis_df[CONCEPT_IDX_COL] == concept_idx
        ].copy()

        concept_name_values = concept_df[CONCEPT_NAME_COL].dropna().unique()
        concept_name = (
            concept_name_values[0]
            if len(concept_name_values)
            else f"concept_{int(concept_idx):02d}"
        )

        concept_df[TRUE_CONCEPT_COL] = pd.to_numeric(
            concept_df[TRUE_CONCEPT_COL],
            errors="coerce",
        )
        concept_df[PRED_CONCEPT_COL] = pd.to_numeric(
            concept_df[PRED_CONCEPT_COL],
            errors="coerce",
        )

        aligned = (
            concept_df[
                np.isclose(concept_df[TRUE_CONCEPT_COL], 1.0)
            ]
            .sort_values(PRED_CONCEPT_COL, ascending=False)
            .head(N_PER_ROW)
            .copy()
        )

        contrastive = (
            concept_df[
                np.isclose(concept_df[TRUE_CONCEPT_COL], 0.0)
            ]
            .sort_values(PRED_CONCEPT_COL, ascending=False)
            .head(N_PER_ROW)
            .copy()
        )

        n_aligned = len(aligned)
        n_contrastive = len(contrastive)

        summary_record = {
            "axis": axis,
            "concept_idx": int(concept_idx),
            "concept_name": concept_name,
            "available_target_1": int(
                np.isclose(concept_df[TRUE_CONCEPT_COL], 1.0).sum()
            ),
            "available_target_0": int(
                np.isclose(concept_df[TRUE_CONCEPT_COL], 0.0).sum()
            ),
            "selected_target_1": n_aligned,
            "selected_target_0": n_contrastive,
            "status": "exported",
            "panel_path": None,
        }

        if n_aligned < N_PER_ROW or n_contrastive < N_PER_ROW:
            summary_record["status"] = "insufficient_endpoint_records"
            panel_summary_records.append(summary_record)

            print(
                f"SKIPPED | {axis:6s} | {concept_name:25s} | "
                f"target=1: {n_aligned}/{N_PER_ROW}, "
                f"target=0: {n_contrastive}/{N_PER_ROW}"
            )
            continue

        panel_rows = [
            ("Target 1: highest predicted concept activations", aligned),
            ("Target 0: highest predicted concept activations", contrastive),
        ]

        fig, axes = plt.subplots(
            nrows=2,
            ncols=N_PER_ROW,
            figsize=(
                FIGSIZE_PER_TILE[0] * N_PER_ROW,
                FIGSIZE_PER_TILE[1] * 2,
            ),
            constrained_layout=True,
        )

        fig.suptitle(
            f"{axis.capitalize()} axis — Concept {int(concept_idx):02d}: "
            f"{concept_name}",
            fontsize=17,
            fontweight="bold",
        )

        for row_idx, (row_label, row_df) in enumerate(panel_rows):
            for col_idx, (_, record) in enumerate(row_df.iterrows()):
                ax = axes[row_idx, col_idx]

                image_value = record.get(IMAGE_PATH_COL, None)
                mask_value = record.get(SUPPORT_MASK_COL, None)

                if pd.isna(image_value) or image_value is None:
                    raise ValueError(
                        f"Missing image path for axis={axis}, "
                        f"concept={concept_name}, row index={record.name}"
                    )

                if pd.isna(mask_value) or mask_value is None:
                    raise ValueError(
                        f"Missing support-mask path for axis={axis}, "
                        f"concept={concept_name}, row index={record.name}. "
                        f"Check SUPPORT_MASK_COL={SUPPORT_MASK_COL!r}."
                    )

                image_path = Path(image_value)
                mask_path = Path(mask_value)

                try:
                    image = load_rgb_image(image_path)
                    support_mask = load_mask(mask_path, image.size)
                    overlay = red_support_overlay(
                        image=image,
                        support_mask=support_mask,
                        alpha=OVERLAY_ALPHA,
                    )

                    ax.imshow(overlay)

                    pred = float(record[PRED_CONCEPT_COL])
                    true = float(record[TRUE_CONCEPT_COL])

                    ax.set_title(
                        f"target={true:.0f} | pred={pred:.3f}",
                        fontsize=10,
                    )

                    panel_export_records.append({
                        "axis": axis,
                        "concept_idx": int(concept_idx),
                        "concept_name": concept_name,
                        "row_condition": (
                            "aligned_target_1"
                            if row_idx == 0
                            else "contrastive_target_0"
                        ),
                        "rank_within_row": col_idx + 1,
                        "true_concept": true,
                        "predicted_concept": pred,
                        "image_path": str(image_path),
                        "support_mask_path": str(mask_path),
                    })

                except Exception as exc:
                    ax.text(
                        0.5,
                        0.5,
                        f"Could not load tile\n{type(exc).__name__}",
                        ha="center",
                        va="center",
                        fontsize=10,
                    )
                    print(
                        f"WARNING | {axis} | {concept_name} | "
                        f"{image_path.name} | {exc}"
                    )

                ax.axis("off")

            axes[row_idx, 0].set_ylabel(
                row_label,
                fontsize=12,
                fontweight="bold",
                rotation=90,
                labelpad=20,
            )

        output_dir = PANEL_ROOT / safe_name(axis)
        output_dir.mkdir(parents=True, exist_ok=True)

        output_path = output_dir / (
            f"concept_{int(concept_idx):02d}_"
            f"{safe_name(concept_name)}_"
            f"target1_vs_target0.png"
        )

        fig.savefig(
            output_path,
            dpi=200,
            bbox_inches="tight",
            facecolor="white",
        )
        plt.close(fig)

        summary_record["panel_path"] = str(output_path)
        panel_summary_records.append(summary_record)

        print(
            f"EXPORTED | {axis:6s} | "
            f"{concept_name:25s} | {output_path.name}"
        )

panel_export_registry = pd.DataFrame(panel_export_records)
panel_export_summary = pd.DataFrame(panel_summary_records)

PANEL_ROOT.mkdir(parents=True, exist_ok=True)

panel_export_registry.to_parquet(
    PANEL_ROOT / "panel_export_registry.parquet",
    index=False,
)

panel_export_summary.to_csv(
    PANEL_ROOT / "panel_export_summary.csv",
    index=False,
)

print("\n" + "=" * 72)
print("PANEL EXPORT SUMMARY")
print("=" * 72)
print(f"Tiles exported: {len(panel_export_registry)}")
print(
    "Panels exported:",
    int((panel_export_summary["status"] == "exported").sum())
)
print(
    "Panels skipped due to insufficient target-0/target-1 records:",
    int(
        (
            panel_export_summary["status"]
            == "insufficient_endpoint_records"
        ).sum()
    )
)

EXPORTED | breadth | contemporary              | concept_00_contemporary_target1_vs_target0.png
EXPORTED | breadth | figurative                | concept_01_figurative_target1_vs_target0.png
EXPORTED | breadth | geometrical               | concept_02_geometrical_target1_vs_target0.png
EXPORTED | breadth | pattern                   | concept_03_pattern_target1_vs_target0.png
EXPORTED | breadth | brushstrokes              | concept_04_brushstrokes_target1_vs_target0.png
EXPORTED | breadth | fixed-pallete             | concept_05_fixed-pallete_target1_vs_target0.png
EXPORTED | breadth | contour                   | concept_06_contour_target1_vs_target0.png
EXPORTED | breadth | focused                   | concept_08_focused_target1_vs_target0.png
EXPORTED | breadth | minimalism                | concept_09_minimalism_target1_vs_target0.png
EXPORTED | breadth | texture                   | concept_10_texture_target1_vs_target0.png
EXPORTED | breadth | isometric                 | concept_11_isom

## Activation Maximization

In [38]:
from pathlib import Path
import math
import re

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torchvision.utils import save_image

# ------------------------------------------------------------------
# Configuration
# ------------------------------------------------------------------

AM_OUT_ROOT = Path("visualizations/googlenet/cbm_activation_maximization")

AM_CONFIG = {
    "image_size": 256,
    "steps": 512,
    "lr": 0.05,
    "tv_weight": 2e-4,
    "l2_weight": 1e-6,
    "jitter_px": 8,
    "seeds": [42, 43, 44],
}

# These are appropriate if the CBM DataLoaders use ImageNet normalization.
# If you used a different Normalize transform while constructing the
# DataLoaders, replace these with its mean and std values.
mean = torch.tensor(imagenet_stats[0], device=device).view(1, 3, 1, 1)
std = torch.tensor(imagenet_stats[1], device=device).view(1, 3, 1, 1)


# ------------------------------------------------------------------
# Utility functions
# ------------------------------------------------------------------

def safe_filename(value):
    value = str(value).strip().lower()
    value = re.sub(r"\s+", "-", value)
    value = re.sub(r"[^a-z0-9_-]+", "", value)
    return value


def normalize_for_model(x_01):
    return (x_01 - mean) / std


def total_variation(x):
    vertical = (x[:, :, 1:, :] - x[:, :, :-1, :]).pow(2).mean()
    horizontal = (x[:, :, :, 1:] - x[:, :, :, :-1]).pow(2).mean()
    return vertical + horizontal


def random_jitter(x, max_shift):
    if max_shift <= 0:
        return x

    shift_y = torch.randint(
        -max_shift, max_shift + 1, (1,), device=x.device
    ).item()
    shift_x = torch.randint(
        -max_shift, max_shift + 1, (1,), device=x.device
    ).item()

    return torch.roll(x, shifts=(shift_y, shift_x), dims=(2, 3))


def concept_logit_from_score(concept_score, eps=1e-5):
    concept_score = concept_score.clamp(eps, 1.0 - eps)
    return torch.logit(concept_score)


@torch.no_grad()
def render_tensor(x_01):
    return x_01.detach().clamp(0, 1).cpu().squeeze(0)


# ------------------------------------------------------------------
# Single-concept activation maximization
# ------------------------------------------------------------------

def optimize_concept_activation(
    model,
    concept_idx,
    image_size=256,
    steps=450,
    lr=0.075,
    tv_weight=2.0e-4,
    l2_weight=1.0e-6,
    jitter_px=8,
    seed=666,
):
    """
    Produces one RGB image in [0, 1] that maximizes one CBM concept score.

    Returns
    -------
    image_01 : torch.Tensor
        Shape [3, H, W], in displayable RGB range [0, 1].
    history : dict
        Optimization trajectory and final concept score/logit.
    """

    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    model = model.eval()

    for parameter in model.parameters():
        parameter.requires_grad_(False)

    # Unconstrained latent image; sigmoid maps it to [0, 1].
    latent = torch.randn(
        1,
        3,
        image_size,
        image_size,
        device=device,
    ) * 0.15

    latent = torch.nn.Parameter(latent.detach())
    optimizer = torch.optim.Adam([latent], lr=lr)

    optimizer = torch.optim.Adam([latent], lr=lr)

    score_trace = []
    logit_trace = []
    loss_trace = []

    for step in range(steps):
        optimizer.zero_grad(set_to_none=True)

        image_01 = torch.sigmoid(latent)
        augmented_01 = random_jitter(image_01, jitter_px)
        model_input = normalize_for_model(augmented_01)

        _, concept_scores = model(model_input)

        target_score = concept_scores[0, concept_idx]
        target_logit = concept_logit_from_score(target_score)

        tv_loss = total_variation(image_01)
        l2_loss = image_01.pow(2).mean()

        loss = (
            -target_logit
            + tv_weight * tv_loss
            + l2_weight * l2_loss
        )

        loss.backward()
        optimizer.step()

        score_trace.append(float(target_score.detach().cpu()))
        logit_trace.append(float(target_logit.detach().cpu()))
        loss_trace.append(float(loss.detach().cpu()))

    final_image = torch.sigmoid(latent)

    with torch.no_grad():
        final_input = normalize_for_model(final_image)
        _, final_scores = model(final_input)

        final_score = float(final_scores[0, concept_idx].cpu())
        final_logit = float(
            concept_logit_from_score(final_scores[0, concept_idx]).cpu()
        )

    return render_tensor(final_image), {
        "concept_score_trace": score_trace,
        "concept_logit_trace": logit_trace,
        "loss_trace": loss_trace,
        "final_concept_score": final_score,
        "final_concept_logit": final_logit,
    }


# ------------------------------------------------------------------
# Full two-axis exporter
# ------------------------------------------------------------------

activation_max_registry = []

for axis, learn in learners.items():
    model = learn.model.eval().to(device)

    axis_dir = AM_OUT_ROOT / axis
    axis_dir.mkdir(parents=True, exist_ok=True)

    for concept_idx, concept_name in enumerate(concept_labels):
        for seed in AM_CONFIG["seeds"]:
            concept_slug = safe_filename(concept_name)

            stem = (
                f"concept-{concept_idx:02d}_{concept_slug}"
                f"_activation-max_score-{history['final_concept_score']:.5f}"
                f"_seed-{seed}"
            )

            image_path = axis_dir / f"{stem}.png"
            metadata_path = axis_dir / f"{stem}.json"

            if image_path.exists() and metadata_path.exists():
                activation_max_registry.append({
                    "axis": axis,
                    "concept_idx": concept_idx,
                    "concept_name": concept_name,
                    "visualization_type": "activation_maximization",
                    "image_path": str(out_path),
                    "steps": AM_CONFIG["steps"],
                    "lr": AM_CONFIG["lr"],
                    "tv_weight": AM_CONFIG["tv_weight"],
                    "l2_weight": AM_CONFIG["l2_weight"],
                    "jitter_px": AM_CONFIG["jitter_px"],
                    "final_concept_score": None,
                    "final_concept_logit": None,
                })
                continue
            
            image_01, history = optimize_concept_activation(
                model=model,
                concept_idx=concept_idx,
                image_size=AM_CONFIG["image_size"],
                steps=AM_CONFIG["steps"],
                lr=AM_CONFIG["lr"],
                tv_weight=AM_CONFIG["tv_weight"],
                l2_weight=AM_CONFIG["l2_weight"],
                jitter_px=AM_CONFIG["jitter_px"],
                seed=seed + concept_idx,
            )

            save_image(image_01, image_path)

            metadata = {
                "visualization_type": "regularized_channel_activation_maximization",
                "seed": seed + concept_idx,
                "focus": axis,
                "concept_idx": concept_idx,
                "concept_name": concept_name,
                "feature_map_shape": 8,
                "image_size": AM_CONFIG["image_size"],
                "lr": AM_CONFIG["lr"],
                "tv_weight": AM_CONFIG["tv_weight"],
                "l2_weight": AM_CONFIG["l2_weight"],
                "jitter_px": AM_CONFIG["jitter_px"],
                "concept_score_trace": history["concept_score_trace"],
                "concept_logit_trace": history["concept_logit_trace"],
                "loss_trace": history["loss_trace"],
                "final_concept_score": history["final_concept_score"] ,
                "final_concept_logit": history["final_concept_logit"] ,
            }

            with open(metadata_path, "w", encoding="utf-8") as file:
                json.dump(metadata, file, indent=2)

            activation_max_registry.append({
                "axis": axis,
                "concept_idx": concept_idx,
                "concept_name": concept_name,
                "visualization_type": "activation_maximization",
                "image_path": str(out_path),
                "steps": AM_CONFIG["steps"],
                "lr": AM_CONFIG["lr"],
                "tv_weight": AM_CONFIG["tv_weight"],
                "l2_weight": AM_CONFIG["l2_weight"],
                "jitter_px": AM_CONFIG["jitter_px"],
                "final_concept_score": history["final_concept_score"],
                "final_concept_logit": history["final_concept_logit"],
            })

            print(
                f"{axis:6s} | {concept_idx:02d} | "
                f"{concept_name:30s} | "
                f"score={history['final_concept_score']:.5f}"
            )

activation_max_registry = pd.DataFrame(activation_max_registry)

registry_path = (
    AM_OUT_ROOT / "cbm_concept_activation_maximization_registry.parquet"
)
activation_max_registry.to_parquet(registry_path, index=False)

print(f"\nSaved {len(activation_max_registry)} synthesized images.")
print(f"Registry: {registry_path}")

breadth | 00 | contemporary                   | score=1.00000
breadth | 00 | contemporary                   | score=0.99999
breadth | 00 | contemporary                   | score=0.99999
breadth | 01 | figurative                     | score=1.00000
breadth | 01 | figurative                     | score=1.00000
breadth | 01 | figurative                     | score=1.00000
breadth | 02 | geometrical                    | score=1.00000
breadth | 02 | geometrical                    | score=1.00000
breadth | 02 | geometrical                    | score=1.00000
breadth | 03 | pattern                        | score=0.99999
breadth | 03 | pattern                        | score=1.00000
breadth | 03 | pattern                        | score=1.00000
breadth | 04 | brushstrokes                   | score=1.00000
breadth | 04 | brushstrokes                   | score=1.00000
breadth | 04 | brushstrokes                   | score=1.00000
breadth | 05 | fixed-pallete                  | score=0.99996
breadth 

In [40]:
from pathlib import Path
import json

root = Path("visualizations/googlenet/cbm_activation_maximization")

pngs = sorted(root.rglob("*.png"))
jsons = sorted(root.rglob("*.json"))

print("PNG files :", len(pngs))
print("JSON files:", len(jsons))
print("Expected  :", 2 * len(concept_labels) * 3)

missing_json = [
    png for png in pngs
    if not png.with_suffix(".json").exists()
]

print("PNGs without JSON sidecar:", len(missing_json))

for path in jsons[:3]:
    with open(path, "r", encoding="utf-8") as f:
        metadata = json.load(f)

    print("\n", path.name)
    print({
        key: metadata.get(key)
        for key in [
            "axis",
            "concept_idx",
            "concept_name",
            "seed",
            "final_concept_score",
            "final_concept_logit",
        ]
    })

PNG files : 102
JSON files: 102
Expected  : 102
PNGs without JSON sidecar: 0

 concept-00_contemporary_activation-max_score-0.99999_seed-44.json
{'axis': None, 'concept_idx': 0, 'concept_name': 'contemporary', 'seed': 44, 'final_concept_score': 0.9999943971633911, 'final_concept_logit': 11.511558532714844}

 concept-00_contemporary_activation-max_score-1.00000_seed-42.json
{'axis': None, 'concept_idx': 0, 'concept_name': 'contemporary', 'seed': 42, 'final_concept_score': 0.9999951124191284, 'final_concept_logit': 11.511558532714844}

 concept-00_contemporary_activation-max_score-1.00000_seed-43.json
{'axis': None, 'concept_idx': 0, 'concept_name': 'contemporary', 'seed': 43, 'final_concept_score': 0.9999901056289673, 'final_concept_logit': 11.511558532714844}


In [3]:
from pathlib import Path
import cv2
import numpy as np
from PIL import Image

POSTPROC_CONFIG = {
    "bilateral_d": 7,
    "bilateral_sigma_color": 45,
    "bilateral_sigma_space": 45,
    "clahe_clip_limit": 1.5,
    "clahe_tile_grid_size": (4, 4),
    "unsharp_sigma": 0.5,
    "unsharp_amount": 0.15,
    "low_frequency_sigma": 1.5,
}

def structure_emphasis_rgb(rgb, cfg=POSTPROC_CONFIG):
    """Fixed display-only transform; input and output are uint8 RGB."""
    rgb = np.asarray(rgb, dtype=np.uint8)

    smooth = cv2.bilateralFilter(
        rgb,
        d=cfg["bilateral_d"],
        sigmaColor=cfg["bilateral_sigma_color"],
        sigmaSpace=cfg["bilateral_sigma_space"],
    )

    lab = cv2.cvtColor(smooth, cv2.COLOR_RGB2LAB)
    l_channel, a_channel, b_channel = cv2.split(lab)

    clahe = cv2.createCLAHE(
        clipLimit=cfg["clahe_clip_limit"],
        tileGridSize=cfg["clahe_tile_grid_size"],
    )
    enhanced_l = clahe.apply(l_channel)

    enhanced = cv2.cvtColor(
        cv2.merge([enhanced_l, a_channel, b_channel]),
        cv2.COLOR_LAB2RGB,
    )

    blurred = cv2.GaussianBlur(
        enhanced,
        ksize=(0, 0),
        sigmaX=cfg["unsharp_sigma"],
    )

    sharpened = cv2.addWeighted(
        enhanced,
        1.0 + cfg["unsharp_amount"],
        blurred,
        -cfg["unsharp_amount"],
        0,
    )

    return np.clip(sharpened, 0, 255).astype(np.uint8)

def low_frequency_rgb(rgb, cfg=POSTPROC_CONFIG):
    """Global colour/mass view; display-only transform."""
    return cv2.GaussianBlur(
        np.asarray(rgb, dtype=np.uint8),
        ksize=(0, 0),
        sigmaX=cfg["low_frequency_sigma"],
    )

def export_activation_max_display_views(root, cfg=POSTPROC_CONFIG):
    root = Path(root)
    raw_paths = sorted(
        p for p in root.rglob("*_activation-max*.png")
        if "_structure-emphasis" not in p.stem
        and "_low-frequency" not in p.stem
    )

    exported = []

    for raw_path in raw_paths:
        rgb = np.array(Image.open(raw_path).convert("RGB"))

        structure_path = raw_path.with_name(
            raw_path.stem + "_structure-emphasis.png"
        )
        low_freq_path = raw_path.with_name(
            raw_path.stem + "_low-frequency.png"
        )

        Image.fromarray(structure_emphasis_rgb(rgb, cfg)).save(structure_path)
        Image.fromarray(low_frequency_rgb(rgb, cfg)).save(low_freq_path)

        exported.append({
            "raw_path": str(raw_path),
            "structure_emphasis_path": str(structure_path),
            "low_frequency_path": str(low_freq_path),
        })

    return exported

display_views = export_activation_max_display_views(
    "visualizations/googlenet/cbm_activation_maximization"
)

print("Raw prototypes processed:", len(display_views))
print("Derived structure views :", len(display_views))
print("Derived low-frequency views:", len(display_views))

Raw prototypes processed: 102
Derived structure views : 102
Derived low-frequency views: 102
